# Crunchbase Data Pipeline
*APAN5400, Columbia University*

Cleans and filters a 1.1M-row Crunchbase dataset (Pandas), validates a JSON round-trip, and serves results via a Flask API.

In [2]:
   from google.colab import drive
   drive.mount('/content/drive')


Mounted at /content/drive


## Task 1: Load and Clean Data
Load the Crunchbase organizations CSV into a Pandas DataFrame, then remove rows with missing company names and duplicate company names.

In [3]:
import pandas as pd

csv_path = "/content/drive/MyDrive/Colab Notebooks/crunchbase_odm_orgs.csv"

df_raw = pd.read_csv(csv_path)
print(df_raw.shape)
print(df_raw.columns.tolist())
df_raw.head()

(1127658, 16)
['uuid', 'name', 'type', 'primary_role', 'cb_url', 'domain', 'homepage_url', 'logo_url', 'facebook_url', 'twitter_url', 'linkedin_url', 'combined_stock_symbols', 'city', 'region', 'country_code', 'short_description']


,uuid,name,type,primary_role,cb_url,domain,homepage_url,logo_url,facebook_url,twitter_url,linkedin_url,combined_stock_symbols,city,region,country_code,short_description
0,e1393508-30ea-8a36-3f96-dd3226033abd,Wetpaint,organization,company,https://www.crunchbase.com/organization/wetpai...,wetpaint.com,http://www.wetpaint.com/,https://res.cloudinary.com/crunchbase-producti...,https://www.facebook.com/Wetpaint,https://twitter.com/wetpainttv,https://www.linkedin.com/company/wetpaint,NaN,New York,New York,USA,Wetpaint offers an online social publishing pl...
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Zoho,organization,company,https://www.crunchbase.com/organization/zoho?u...,zoho.com,https://www.zoho.com/,https://res.cloudinary.com/crunchbase-producti...,http://www.facebook.com/zoho,http://twitter.com/zoho,http://www.linkedin.com/company/zoho-corporati...,NaN,Pleasanton,California,USA,"Zoho offers a suite of business, collaboration..."
2,5f2b40b8-d1b3-d323-d81a-b7a8e89553d0,Digg,organization,company,https://www.crunchbase.com/organization/digg?u...,digg.com,http://www.digg.com,https://res.cloudinary.com/crunchbase-producti...,http://www.facebook.com/digg,http://twitter.com/digg,http://www.linkedin.com/company/digg,NaN,New York,New York,USA,Digg Inc. operates a website that enables its ...
3,df662812-7f97-0b43-9d3e-12f64f504fbb,Facebook,organization,company,https://www.crunchbase.com/organization/facebo...,facebook.com,http://www.facebook.com,https://res.cloudinary.com/crunchbase-producti...,https://www.facebook.com/facebook/,https://twitter.com/facebook,http://www.linkedin.com/company/facebook,nasdaq:FB,Menlo Park,California,USA,Facebook is an online social networking servic...
4,b08efc27-da40-505a-6f9d-c9e14247bf36,Accel,organization,investor,https://www.crunchbase.com/organization/accel?...,accel.com,http://www.accel.com,https://res.cloudinary.com/crunchbase-producti...,http://www.facebook.com/accel,http://twitter.com/accel,https://www.linkedin.com/company/accel-vc/,NaN,Palo Alto,California,USA,Accel is an early and growth-stage venture cap...


In [4]:
df = pd.read_csv(csv_path)
print("Rows loaded:", len(df))

Rows loaded: 1127658


In [5]:
df = df.dropna(subset=["name"])
print("After dropping null company names:", len(df))

After dropping null company names: 1127657


In [6]:
df = df.drop_duplicates(subset=["name"], keep="first")
print("After dropping duplicates:", len(df))

After dropping duplicates: 1100481


## Task 2: Filter and Transform
Filter for USA-based companies whose name starts with "ad" (case-insensitive), then add a new `name_length` column.

In [7]:
usa_companies = df["country_code"] == "USA"
company_name_starts_with_ad = df["name"].str.lower().str.startswith("ad")
df_ad = df[usa_companies & company_name_starts_with_ad].copy()

In [8]:
print(len(df_ad))

2980


In [9]:
df_ad["name_length"] = df_ad["name"].str.len()
print("row count:", len(df_ad))
df_ad.head(10)

row count: 2980


,uuid,name,type,primary_role,cb_url,domain,homepage_url,logo_url,facebook_url,twitter_url,linkedin_url,combined_stock_symbols,city,region,country_code,short_description,name_length
30,fc89cfde-a28e-e12c-443c-edccb6c93fd4,AddThis,organization,company,https://www.crunchbase.com/organization/addthi...,addthis.com,http://www.addthis.com,https://res.cloudinary.com/crunchbase-producti...,http://www.facebook.com/AddThis,http://twitter.com/addthis,http://www.linkedin.com/company/addthis,NaN,Vienna,Virginia,USA,"AddThis provides social engagement tools, APIs...",7
131,9bc6d32d-0bd1-6c54-b7b2-d7436e948bc0,adBrite,organization,company,https://www.crunchbase.com/organization/adbrit...,adbrite.com,http://adbrite.com,https://res.cloudinary.com/crunchbase-producti...,https://www.facebook.com/sitescout,http://twitter.com/adBrite,NaN,NaN,San Francisco,California,USA,"adBrite, an advertising exchange platform, pro...",7
301,172a34b4-9edd-b0ae-c768-cbfa8e1ab52c,Adobe Systems,organization,company,https://www.crunchbase.com/organization/adobe-...,adobe.com,http://www.adobe.com,https://res.cloudinary.com/crunchbase-producti...,http://www.facebook.com/Adobe,http://twitter.com/Adobe,http://www.linkedin.com/company/adobe,nasdaq:ADBE,San Jose,California,USA,Adobe provides digital media and digital marke...,13
479,ab7be664-d0cf-f055-e82f-c7daa1f59e45,Advertising.com,organization,company,https://www.crunchbase.com/organization/advert...,advertising.com,https://www.advertising.com/,https://res.cloudinary.com/crunchbase-producti...,http://www.facebook.com/AOLPlatfoms,http://twitter.com/aolplatforms,https://www.linkedin.com/company/advertising-c...,NaN,New York,New York,USA,"Advertising.com, now a part of ONE by AOL, ena...",15
578,82176a6f-806b-1517-317e-1e318dca8f81,AdMob,organization,company,https://www.crunchbase.com/organization/admob?...,google.com,http://www.google.com/admob,https://res.cloudinary.com/crunchbase-producti...,http://www.facebook.com/pages/Admob/1403745529...,http://twitter.com/admob,http://www.linkedin.com/company/24866,NaN,Mountain View,California,USA,AdMob is a mobile advertising network that hel...,5
822,96f8cb80-0d79-328d-57d1-827f0b620c40,AdReady,organization,company,https://www.crunchbase.com/organization/adread...,adready.com,http://adready.com,https://res.cloudinary.com/crunchbase-producti...,https://www.facebook.com/cpxinteractive,http://twitter.com/AdReady,http://www.linkedin.com/company/adready,NaN,Seattle,Washington,USA,"AdReady, a division of CPXi, provides a self-s...",7
850,79ee09de-d880-0f35-6b2f-820ec3871f42,Adobe Sign,organization,company,https://www.crunchbase.com/organization/echosi...,acrobat.adobe.com,https://acrobat.adobe.com/us/en/sign.html,https://res.cloudinary.com/crunchbase-producti...,https://www.facebook.com/adobedocumentcloud,http://twitter.com/adobedoccloud,https://www.linkedin.com/company/adobe-documen...,NaN,San Francisco,California,USA,"Adobe Sign, formerly EchoSign is an e-signatur...",10
858,76809208-640b-24f2-5f11-72622d07ef52,ADI Time,organization,company,https://www.crunchbase.com/organization/aditim...,aditime.com,http://www.aditime.com,https://res.cloudinary.com/crunchbase-producti...,NaN,https://www.twitter.com/asure_software,https://www.linkedin.com/company/asure-software,NaN,Warwick,Rhode Island,USA,ADI Time develops cloud computing time and Saa...,8
977,d5648438-e3ad-4bbd-96ea-03c0a3238eba,Adteractive,organization,company,https://www.crunchbase.com/organization/adtera...,adteractive.com,http://adteractive.com,https://res.cloudinary.com/crunchbase-producti...,NaN,NaN,NaN,NaN,San Francisco,California,USA,Adteractive is an online lead generation compa...,11
1133,01413ae9-b575-6101-8588-1ed6b0b3fc02,Adaptive Path,organization,company,https://www.crunchbase.com/organization/adapti...,adaptivepath.com,http://www.adaptivepath.com,https://res.cloudinary.com/crunchbase-producti...,http://www.facebook.com/adaptivepath/info,http://twitter.com/AdaptivePath,http://www.linkedin.com/company/adaptive-path,

## Task 3: JSON Conversion and Validation
Convert the filtered DataFrame to JSON, write it to file, then read it back and verify the row counts match to confirm no data was lost in the round-trip.

In [10]:
import json

In [11]:
json_text = df_ad.to_json(orient="records")
json_list = json.loads(json_text)

In [12]:
json_path = "/content/drive/MyDrive/usa_ad_companies.json"
with open(json_path, "w") as json_file:
  json.dump(json_list, json_file)

In [13]:
df_json = pd.read_json(json_path)

In [14]:
row_count = len(df_ad) == len(df_json)

In [15]:
print("JSON object count:", len(json_list))
print("first 5 JSON objects:", json_list[:5])
print("Row counts match:", row_count)

JSON object count: 2980
first 5 JSON objects: [{'uuid': 'fc89cfde-a28e-e12c-443c-edccb6c93fd4', 'name': 'AddThis', 'type': 'organization', 'primary_role': 'company', 'cb_url': 'https://www.crunchbase.com/organization/addthis?utm_source=crunchbase&utm_medium=export&utm_campaign=odm_csv', 'domain': 'addthis.com', 'homepage_url': 'http://www.addthis.com', 'logo_url': 'https://res.cloudinary.com/crunchbase-production/image/upload/v1398228329/athjbh8d9o6fsbeyeo8h.png', 'facebook_url': 'http://www.facebook.com/AddThis', 'twitter_url': 'http://twitter.com/addthis', 'linkedin_url': 'http://www.linkedin.com/company/addthis', 'combined_stock_symbols': None, 'city': 'Vienna', 'region': 'Virginia', 'country_code': 'USA', 'short_description': 'AddThis provides social engagement tools, APIs and services personalizing websites for engagement, monetization and traffic boosting.', 'name_length': 7}, {'uuid': '9bc6d32d-0bd1-6c54-b7b2-d7436e948bc0', 'name': 'adBrite', 'type': 'organization', 'primary_rol

In [16]:
df_ny = df_json[df_json["region"] == "New York"].copy()

In [17]:
print("row count of New York:", len(df_ny))

row count of New York: 261


## Task 4: Serve via Flask
Filter the JSON-sourced data to New York-region companies only, then expose the results as a plain-text list through a Flask API endpoint.

In [18]:
from flask import Flask
from google.colab import output

app = Flask(__name__)

@app.route("/nycompanies")
def list_ny_companies():
      # I chose a plain-text because it's quick and easy to read the company names.
    return str(df_ny["name"].to_list())

output.serve_kernel_port_as_window(5200, path="/nycompanies")
app.run(host="localhost", port=5200)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://localhost:5200
INFO:werkzeug:Press CTRL+C to quit
